In [ ]:
!apt install swig cmake

In [ ]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit1/requirements-unit1.txt

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# import os
# os.kill(os.getpid(), 9)

In [1]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [2]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [3]:
import gymnasium as gym

# First, we create our environment called LunarLander-v2
env = gym.make("LunarLander-v2")

# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
  # Take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 0
Action taken: 3
Action taken: 0
Action taken: 1
Action taken: 3
Action taken: 0
Action taken: 0
Action taken: 3
Action taken: 2
Action taken: 2
Action taken: 2
Action taken: 2
Action taken: 0
Action taken: 1
Action taken: 1
Action taken: 3
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 0


In [4]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("LunarLander-v2")
env.reset()
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [44.50827     3.9831421  -4.314208    2.5531259   0.7102734   2.1941254
  0.9540051   0.92511535]


In [5]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 1


In [6]:
# Create the environment
env = make_vec_env('LunarLander-v2', n_envs=16)

In [7]:
# SOLUTION
# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cuda device


In [8]:
# SOLUTION
# Train it for 1,000,000 timesteps
model.learn(total_timesteps=1000000)
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 91.5     |
|    ep_rew_mean     | -177     |
| time/              |          |
|    fps             | 5545     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 16384    |
---------------------------------
--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 100            |
|    ep_rew_mean          | -144           |
| time/                   |                |
|    fps                  | 3379           |
|    iterations           | 2              |
|    time_elapsed         | 9              |
|    total_timesteps      | 32768          |
| train/                  |                |
|    approx_kl            | 0.005410514    |
|    clip_fraction        | 0.0454         |
|    clip_range           | 0.2            |
|    entropy_loss         | -1.38          |
|    explained_variance   | -0

In [9]:
#@title
eval_env = Monitor(gym.make("LunarLander-v2", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=245.66 +/- 20.480055593214242


In [11]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import PPO

# 1. Load your trained model (Change the path if you saved it with a different name)
model = PPO.load("ppo-LunarLander-v2")

# 2. Create the environment with rgb_array rendering (required for video)
env = gym.make("LunarLander-v2", render_mode="rgb_array")

# 3. Wrap the environment to record videos
# episode_trigger=lambda x: True means "record every episode"
env = RecordVideo(
    env, 
    video_folder="./videos", 
    episode_trigger=lambda x: x < 3, # Record the first 3 episodes
    name_prefix="lunarlander_play"
)

# 4. Run the agent
for episode in range(3):
    obs, info = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # The model predicts the best action based on the observation
        action, _ = model.predict(obs, deterministic=True)
        
        # Take the action in the environment
        obs, reward, terminated, truncated, info = env.step(action)
        
        total_reward += reward
        done = terminated or truncated
        
    print(f"Episode {episode + 1} finished with total reward: {total_reward}")

env.close()
print("Videos saved in the ./videos folder!")

/home/mojtaba/miniconda3/envs/RL/lib/python3.12/site-packages/gymnasium/wrappers/record_video.py:87: UserWarning: WARN: Overwriting existing videos at /home/mojtaba/Desktop/RLCourse/HuggingFace/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-0.mp4.
MoviePy - Writing video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-0.mp4
Episode 1 finished with total reward: 288.24714843342326
MoviePy - Building video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-1.mp4.
MoviePy - Writing video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-1.mp4



MoviePy - Done !
MoviePy - video ready /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-1.mp4
Episode 2 finished with total reward: 274.19724772924974
MoviePy - Building video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-2.mp4.
MoviePy - Writing video /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-2.mp4



MoviePy - Done !
MoviePy - video ready /home/mojtaba/Desktop/RLCourse/HuggingFace/videos/lunarlander_play-episode-2.mp4
Episode 3 finished with total reward: 247.19851999823848
Videos saved in the ./videos folder!
